In [1]:
import json
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
import requests

from falcon_formation.data_operations import load_config, save_team_data_from_csv

In [2]:
save_team_data_from_csv("data/FALCONS_1.csv")

Players with missing skill level:
17    Jamie Nagele (Goalie)
29          Marc  Zangrando
Name: name, dtype: object
Players with missing positions:
17    Jamie Nagele (Goalie)
29          Marc  Zangrando
Name: name, dtype: object


In [3]:
team_name = "falcons_1"
config_path = ".env"
team_id, activity_name, auth, encryption_key = load_config(config_path, team_name)
# date = "2024-01-01"

In [4]:
import subprocess

result = subprocess.run(
    [
        "/opt/homebrew/bin/openssl",
        "enc",
        "-aes-256-cbc",
        "-salt",
        "-in",
        "data/FALCONS_1.json",
        "-out",
        "data/FALCONS_1.json.enc",
        "-pass",
        f"pass:{encryption_key}",
        "-pbkdf2",
    ],
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

In [ ]:
url = f"https://api.holdsport.dk/v1/teams/{team_id}/activities?date={date}page=1&per_page=150"
headers = {"Accept": "application/json"}

response = requests.get(url, headers=headers, auth=auth, timeout=30)
response_dict = json.loads(response.text)

In [ ]:
def get_attending_player_number(activities_users: list[dict[str, any]]) -> int:  # noqa: D103
    attending_players = 0
    for player in activities_users:
        if player["status"] == "Attending":
            attending_players += 1
    return attending_players

In [ ]:
data = []

for response_entry in response_dict:
    start_time = str(response_entry["starttime"])

    if response_entry["name"] == activity_name:
        entry = {
            "date": datetime.strptime(start_time, "%Y-%m-%dT%H:%M:%S%z").date(),
            "attending": get_attending_player_number(response_entry["activities_users"]),
        }
        data.append(entry)

In [ ]:
df = pd.DataFrame(data)  # noqa: PD901
df.set_index("date", inplace=True)  # noqa: PD002

In [ ]:
df["attending"].plot.bar()
plt.ylabel("Number of registered players")
plt.grid(axis="y")
plt.show()

In [ ]:
def increase_attending_player_numbers(  # noqa: D103
    attended_players: dict[str, int],
    activities_users: list[dict[str, any]],
) -> dict[str, int]:
    for player in activities_users:
        if player["status"] == "Attending":
            if player["name"] not in attended_players:
                attended_players[player["name"]] = 1
            else:
                attended_players[player["name"]] += 1
    return attended_players

In [ ]:
attending_player_numbers = {}

for response_entry in response_dict:
    if response_entry["name"] == activity_name:
        attended_players = increase_attending_player_numbers(
            attending_player_numbers,
            response_entry["activities_users"],
        )

In [ ]:
attending_player_numbers = dict(sorted(attended_players.items(), key=lambda item: item[1], reverse=True))
attending_player_numbers